# UAV-to-satellite baseline

A naive concat zero-shot DiffusionSat baseline.

---

**TL;DR**

Zero-shot UAV-to-satellite retrieval with DiffusionSat UNet on the VisLoc dataset.

Gallery: 256×256 satellite image chunks extracted at scale=0.2 from VisLoc flight 03.
Query: UAV drone images from the same flight, matched to satellite chunks by GPS containment.

Notes:

- This zero-shot setup achieves ~2% R@1. Domain shift is significant
- t-SNE visualization shows two independent clusters of UAV and satellite imagery


## Setup


In [4]:
%load_ext autoreload
%autoreload 2

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

sys.path.append("..")

warnings.filterwarnings(
  "ignore", category=SyntaxWarning, message=".*invalid escape sequence.*"
)

DEVICE = torch.device("cuda")
DTYPE = torch.bfloat16
VISLOC_ROOT = Path(os.environ["VISLOC_ROOT"])
DIFFUSIONSAT_256_CHCKPT = Path(os.environ["DIFFUSIONSAT_256_CHCKPT"])

FLIGHT_ID = "03"
SAT_SCALE = 0.25
CHUNK_SIZE = 512
CHUNK_STRIDE = 128
BATCH_SIZE = 128


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Model


In [6]:
from src.backbone import DiffusionSatBackbone

backbone = DiffusionSatBackbone(DIFFUSIONSAT_256_CHCKPT, DEVICE, DTYPE)


/workspace/diffusion-vpr/.venv/lib/python3.10/site-packages/accelerate/utils/torch_xla.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/workspace/diffusion-vpr/.venv/lib/python3.10/site-packages/diffusers/models/cross_attention.py:30: FutureWarning: Importing from cross_attention is deprecated. Please import from diffusers.models.attention_processor instead.
  deprecate(
Some weights of the model checkpoint at /workspace/checkpoints/finetune_sd21_256_sn-satlas-fmow_snr5_md7norm_bs64_trimmed were not used when initializing SatUNet: ['metadata_embedding.1.linear_1.weight', 'metadata_embedding.5.linear_2.weight', 'metadata_embedding.5.linear_1.weight', 'metadata_embedding.2.linear_1.weight', 'metadata_embedding.3.linear_1.bias', 'metadata_embedding.2.linear_2.bias', 'm

## Data


In [7]:
from src.datasets.visloc import (
  SatChunkDataset,
  UAVDataset,
  inference_sat_transforms,
  inference_uav_transforms,
)

gallery_dataset = SatChunkDataset(
  VISLOC_ROOT,
  FLIGHT_ID,
  chunk_pixels=CHUNK_SIZE,
  stride_pixels=CHUNK_STRIDE,
  scale_factor=SAT_SCALE,
  transform=inference_sat_transforms,
)
gallery_loader = torch.utils.data.DataLoader(
  gallery_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

uav_dataset = UAVDataset(VISLOC_ROOT, FLIGHT_ID, transform=inference_uav_transforms)
uav_loader = torch.utils.data.DataLoader(
  uav_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True
)

print(f"Gallery: {len(gallery_dataset)} satellite chunks")
print(f"Query:   {len(uav_dataset)} UAV images")


Gallery: 2860 satellite chunks
Query:   768 UAV images


## Main

Perform UAV-to-satellite retrieval: given a UAV drone image, retrieve the satellite map
tile that corresponds to the same geographic location.

Gallery: satellite image chunks from VisLoc flight 03 at scale=0.2 (256×256 tiles, stride=128).
Query: UAV drone images from the same flight.


In [8]:
# Precompute VAE latents for gallery and query. Done once; reused across all grid configs.


def precompute_latents(loader, desc="Latents"):
  latents_list, lats_list, lons_list = [], [], []
  with torch.inference_mode():
    g = torch.Generator(device=DEVICE).manual_seed(0)
    for imgs, lats, lons in tqdm(loader, desc=desc):
      imgs = imgs.to(DEVICE, dtype=DTYPE)
      latents_list.append(backbone.vae.encode(imgs).latent_dist.sample(generator=g) * 0.18215)
      lats_list.append(lats.numpy())
      lons_list.append(lons.numpy())
  return latents_list, np.concatenate(lats_list), np.concatenate(lons_list)


gallery_latents, gallery_lats, gallery_lons = precompute_latents(gallery_loader, "Gallery latents")
uav_latents, uav_lats, uav_lons = precompute_latents(uav_loader, "UAV latents")


UAV latents: 100%|██████████| 6/6 [00:31<00:00,  5.24s/it]


In [9]:
# Build GPS-based ground truth: for each UAV image, find satellite chunks whose bbox contains it.

from src.evaluation import build_ground_truth

uav_coords = np.stack([uav_lats, uav_lons], axis=1)
ground_truth = build_ground_truth(uav_coords, gallery_dataset.chunk_bboxes)
print(f"{len(ground_truth)} UAV queries, avg {np.mean([len(gt) for gt in ground_truth]):.1f} matching chunks each")


768 UAV queries, avg 15.9 matching chunks each


In [ ]:
# Grid search over timesteps and layer configs.

from itertools import product

from src.embedders import PoolConcatEmbedder, normalize_embeddings
from src.evaluation import calculate_metrics
from src.ldm_extractor import LDMExtractorCfg
from src.retrievers import FAISSRetriever


def extract_embeddings(latents_list, embedder):
  embeddings = []
  with torch.inference_mode():
    for latents_batch in tqdm(latents_list, desc="Extracting features"):
      batch_feats, _ = backbone.ldm_extractor.forward(latents_batch)
      batch_embs = embedder(batch_feats)
      embeddings.append(batch_embs.cpu())
  return normalize_embeddings(torch.cat(embeddings, dim=0))


grid = {
  # Which diffusion timesteps to extract features from.
  # Larger index = noisier, smaller = closer to clean image.
  "save_timesteps": [
    [49, 48, 47],
    [49, 48],
    [49],
    [47],
    [42],
    [48, 46, 42],
  ],
  # Which SatUNet layers to extract features from.
  "layer_idxs": [
    {"down_blocks": {"attn1": "all"}},
    # {"up_blocks": {"attn1": "all"}},
  ],
}

results = []
configs = list(product(grid["save_timesteps"], grid["layer_idxs"]))

for idx, (save_timesteps, layer_idxs) in enumerate(configs):
  print(f"Grid search step {idx+1}/{len(configs)}: save_timesteps={save_timesteps}, layer_idxs={layer_idxs}")

  print("Setting extractor...")
  cfg = LDMExtractorCfg(save_timesteps=save_timesteps, layer_idxs=layer_idxs)
  backbone.set_ldm_extractor_cfg(cfg)

  print("Creating embedder...")
  embedder = PoolConcatEmbedder(
    feature_dims=backbone.ldm_extractor.collected_dims,
    save_timesteps=save_timesteps,
  )

  print("Extracting gallery embeddings...")
  gallery_embs = extract_embeddings(gallery_latents, embedder)
  print("Extracting uav embeddings...")
  uav_embs = extract_embeddings(uav_latents, embedder)

  retriever = FAISSRetriever(gallery_embs)
  _, preds = retriever.search(uav_embs, k=10)

  print("Evaluating...")
  metrics = calculate_metrics(preds, ground_truth)
  results.append({
    "save_timesteps": str(save_timesteps),
    "num_timesteps": cfg.num_timesteps,
    "layer_idxs": str(layer_idxs),
    "emb_dim": embedder.embedding_dim,
    "n_gallery": len(gallery_dataset),
    "n_query": len(uav_dataset),
    "normalize_embeddings": True,
    **metrics,
  })
  print(f"  {save_timesteps} | {list(layer_idxs)} → R@1={metrics['Recall@1']:.4f}")

df = pd.DataFrame(results).sort_values("Recall@1", ascending=False)
df


Grid search step 1/5: save_timesteps=[48, 46, 42], layer_idxs={'down_blocks': {'attn1': 'all'}}
Setting extractor...
Creating embedder...
Extracting gallery embeddings...


Extracting features:  13%|█▎        | 3/23 [00:49<05:30, 16.52s/it]


KeyboardInterrupt: 

In [ ]:
df.to_csv("../results/1-baseline-u2s-diffusionsat-naive-results.csv", index=False)
df


In [ ]:
# Run best configuration and compute final embeddings for visualization.

import ast

best = df.iloc[0]
print(f"Best config:")
print(f"  save_timesteps={best['save_timesteps']}")
print(f"  layer_idxs={best['layer_idxs']}")
print(f"  R@1={best['Recall@1']:.4f}  R@5={best['Recall@5']:.4f}  R@10={best['Recall@10']:.4f}")

best_save_timesteps = ast.literal_eval(best["save_timesteps"])
best_layer_idxs = ast.literal_eval(best["layer_idxs"])

cfg = LDMExtractorCfg(save_timesteps=best_save_timesteps, layer_idxs=best_layer_idxs)
backbone.set_ldm_extractor_cfg(cfg)

best_embedder = PoolConcatEmbedder(
  feature_dims=backbone.ldm_extractor.collected_dims,
  save_timesteps=best_save_timesteps,
)

gallery_embs = extract_embeddings(gallery_latents, best_embedder)
uav_embs = extract_embeddings(uav_latents, best_embedder)

retriever = FAISSRetriever(gallery_embs)
distances, preds = retriever.search(uav_embs, k=10)
print(calculate_metrics(preds, ground_truth))


In [ ]:
# fmt: off
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE


def plot_tsne(gallery: torch.Tensor, query: torch.Tensor, ground_truth: list, n_display: int = 300) -> None:
  """t-SNE of gallery (satellite chunks) and query (UAV images) embeddings.
  Lines connect each UAV query to its ground-truth satellite chunk(s) if displayed."""
  rng = np.random.default_rng(42)
  g_idx = rng.choice(len(gallery), min(n_display, len(gallery)), replace=False)
  q_idx = rng.choice(len(query), min(n_display, len(query)), replace=False)

  g_sub = gallery[g_idx].cpu().numpy()
  q_sub = query[q_idx].cpu().numpy()
  all_embs = np.concatenate([g_sub, q_sub])

  coords = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(all_embs)
  ng = len(g_sub)
  g_coords, q_coords = coords[:ng], coords[ng:]

  g_idx_to_pos = {int(orig): pos for pos, orig in enumerate(g_idx)}

  fig, ax = plt.subplots(figsize=(14, 10))

  for q_pos, q_orig in enumerate(q_idx):
    for gt_chunk_idx in ground_truth[int(q_orig)]:
      if gt_chunk_idx in g_idx_to_pos:
        g_pos = g_idx_to_pos[gt_chunk_idx]
        ax.plot(
          [q_coords[q_pos, 0], g_coords[g_pos, 0]],
          [q_coords[q_pos, 1], g_coords[g_pos, 1]],
          color="gray", linewidth=0.5, alpha=0.35, zorder=1,
        )

  ax.scatter(*g_coords.T, c="#4ECDC4", s=60, alpha=0.7, edgecolors="black", linewidth=0.3, zorder=2, label="Gallery (satellite chunk)")
  ax.scatter(*q_coords.T, c="#FF6B6B", s=60, alpha=0.7, edgecolors="black", linewidth=0.3, zorder=2, marker="^", label="Query (UAV image)")

  ax.legend(fontsize=11)
  ax.set_title("t-SNE of DiffusionSat Embeddings (satellite gallery vs UAV queries)", fontsize=14)
  ax.set_xlabel("t-SNE Dimension 1")
  ax.set_ylabel("t-SNE Dimension 2")
  ax.grid(True, alpha=0.3)
  plt.tight_layout()
  plt.show()


plot_tsne(gallery_embs, uav_embs, ground_truth)


In [ ]:
# Visualize top-k satellite retrievals for a selection of UAV queries.
# Hard = queries with lowest cosine similarity to their ground-truth match.

from torchvision import transforms

display_transform = transforms.Compose(
  [
    transforms.Resize(
      (256, 256), interpolation=transforms.InterpolationMode.BILINEAR, antialias=True
    ),
    transforms.ToTensor(),
  ]
)
display_uav = UAVDataset(VISLOC_ROOT, FLIGHT_ID, transform=display_transform)
display_gallery = SatChunkDataset(
  VISLOC_ROOT,
  FLIGHT_ID,
  chunk_pixels=CHUNK_SIZE,
  stride_pixels=CHUNK_STRIDE,
  scale_factor=SAT_SCALE,
  transform=display_transform,
)


def visualize_retrievals(
  preds: np.ndarray,
  ground_truth: list,
  uav_dataset: torch.utils.data.Dataset,
  gallery_dataset: torch.utils.data.Dataset,
  uav_embs: torch.Tensor,
  gallery_embs: torch.Tensor,
  n_queries: int = 6,
  top_k: int = 5,
  mode: str = "hard",  # "hard" | "random"
) -> None:
  """Show UAV queries alongside top-k satellite retrievals. Green border = correct."""
  gt_sims = []
  for i, gt_list in enumerate(ground_truth):
    q_emb = uav_embs[i].numpy()
    g_embs = gallery_embs[list(gt_list)].numpy()
    gt_sims.append(float(np.max(q_emb @ g_embs.T)))
  gt_sims = np.array(gt_sims)

  if mode == "hard":
    query_indices = np.argsort(gt_sims)[:n_queries]
  else:
    query_indices = np.random.default_rng(42).choice(len(uav_dataset), n_queries, replace=False)

  fig, axes = plt.subplots(n_queries, top_k + 1, figsize=(2.5 * (top_k + 1), 2.5 * n_queries))
  fig.suptitle(
    f"UAV query -> Top-{top_k} satellite retrievals  (green=correct, red=incorrect)",
    fontsize=12,
    y=1.01,
  )

  for row, q_idx in enumerate(query_indices):
    uav_img, uav_lat, uav_lon = uav_dataset[q_idx]
    gt_set = set(ground_truth[q_idx])

    ax = axes[row, 0]
    ax.imshow(uav_img.permute(1, 2, 0).numpy())
    ax.set_title(f"UAV {q_idx}\n({uav_lat:.5f}, {uav_lon:.5f})\nsim={gt_sims[q_idx]:.3f}", fontsize=7)
    ax.axis("off")

    for col in range(top_k):
      chunk_idx = int(preds[q_idx, col])
      sat_img_t, sat_lat, sat_lon = gallery_dataset[chunk_idx]
      correct = chunk_idx in gt_set

      ax = axes[row, col + 1]
      ax.imshow(sat_img_t.permute(1, 2, 0).numpy())
      for spine in ax.spines.values():
        spine.set_edgecolor("green" if correct else "red")
        spine.set_linewidth(3)
      ax.set_title(f"#{col + 1} chunk {chunk_idx}\n({sat_lat:.5f}, {sat_lon:.5f})", fontsize=7)
      ax.axis("off")

  plt.tight_layout()
  plt.show()


visualize_retrievals(preds, ground_truth, display_uav, display_gallery, uav_embs, gallery_embs)
